# YoungNAKO · Machine Learning · Hands-on 1 
#### Thomas Küstner, University Hospital of Tübingen
## From regression to deep learning

**Thursday 05.11.2026, Block 1 (≈ 30 min)** · Thomas Küstner, MIDAS.lab, University Hospital Tübingen

In this notebook you will
1. **Part A (≈ 10 min):** predict incident type 2 diabetes in a *synthetic* NAKO-like cohort with four classical ML models and one tabular foundation model (TabPFN)
2. **Part B (≈ 20 min):** train your first **convolutional neural network (CNN)** on real abdominal MRI (CHAOS data) and watch it learn and overfit

**How this works:** you don't need to know Python. Run the cells top to bottom with **Shift + Enter**.
Cells marked **✏️ TODO** contain a blank `___`. Replace it with the right value (hints are in the comments), then run the cell.

> ⚙️ **Before you start:** `Runtime → Change runtime type → T4 GPU`. Everything also runs on CPU, just slower.

In [ ]:
#@title 📦 Setup (≈ 1 min) { display-mode: "form" }
# Install the requirements into the running kernel: requirements.txt of the repo if it is next to
# this notebook (local / cloned repo), otherwise (Colab opened via link) this notebook's packages.
import os
_req = next((p for p in ("requirements.txt", "../requirements.txt") if os.path.isfile(p)), None)
get_ipython().run_line_magic("pip", f"install -q -r {_req}" if _req else
                             "install -q numpy pandas matplotlib scipy scikit-learn torch pillow pydicom SimpleITK")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0); np.random.seed(0)
print("Running on:", DEVICE)

---
# Part A: Tabular data, the classics and a foundation model

We simulate a cohort of 4,000 participants with typical NAKO baseline variables.
The outcome `t2d_5y` (incident type 2 diabetes within 5 years) depends **non-linearly** on the variables, e.g. through an HbA1c threshold and a BMI × age interaction. *The data are fully synthetic; no real participant is involved.*

In [ ]:
#@title 🔧 Simulate the synthetic cohort (run) { display-mode: "form" }
def make_cohort(n=4000, seed=42):
    rng = np.random.default_rng(seed)
    sex = rng.choice(["female", "male"], n)
    age = rng.uniform(20, 72, n)
    centre = rng.choice(["Münster", "Berlin", "Hamburg", "Leipzig", "Freiburg"], n)
    bmi = rng.normal(26.5, 4.5, n) + 0.04 * (age - 45)
    waist = 0.85 * bmi * 3.2 + 6 * (sex == "male") + rng.normal(0, 5, n)
    sbp = 110 + 0.5 * (age - 20) + 0.8 * (bmi - 25) + 5 * (sex == "male") + rng.normal(0, 12, n)
    hba1c = 5.2 + 0.012 * (age - 20) + 0.05 * (bmi - 25) + rng.normal(0, 0.35, n)
    hdl = 58 - 6 * (sex == "male") - 0.6 * (bmi - 25) + rng.normal(0, 10, n)
    ldl = 120 + 0.6 * (age - 45) + rng.normal(0, 30, n)
    smoking = rng.choice(["never", "former", "current"], n, p=[0.5, 0.3, 0.2])
    activity = np.clip(rng.gamma(2.0, 12, n), 0, None)          # MET-h/week
    education = rng.choice(["low", "medium", "high"], n, p=[0.2, 0.5, 0.3])
    family_hist = rng.random(n) < 0.25
    logit = (-4.0 + 0.035 * (age - 45) + 0.10 * (bmi - 26) + 2.8 * np.maximum(hba1c - 5.7, 0)
             + 0.7 * family_hist - 0.02 * (activity - 24) + 0.35 * (smoking == "current")
             + 0.05 * np.maximum(bmi - 30, 0) * np.maximum(age - 45, 0) / 5 - 0.02 * (hdl - 55)
             + 0.3 * (education == "low"))
    y = rng.random(n) < 1 / (1 + np.exp(-logit))
    return pd.DataFrame(dict(age=age.round(1), sex=sex, centre=centre, bmi=bmi.round(1), waist_cm=waist.round(1),
                             sbp=sbp.round(0), hba1c=hba1c.round(2), hdl=hdl.round(0), ldl=ldl.round(0),
                             smoking=smoking, activity_met=activity.round(1), education=education,
                             family_history=family_hist.astype(int), t2d_5y=y.astype(int)))

cohort = make_cohort()
cohort.head()

In [ ]:
print("Participants:", len(cohort), "| incident T2D:", cohort.t2d_5y.sum(), f"({cohort.t2d_5y.mean():.1%})")
cohort.describe().round(1)

### A1 · Train / test split
Exactly as in the lecture: we **fit** on the training data and **evaluate** on data the model has never seen.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X = cohort.drop(columns="t2d_5y"); y = cohort.t2d_5y.values
num_cols = X.select_dtypes("number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()

# ✏️ TODO: keep 25 % of the participants as an independent test set  (hint: a number between 0 and 1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=___, stratify=y, random_state=0)

# One-hot encode categories, standardise numbers. Fitted on the TRAINING data only!
prep = ColumnTransformer([("num", StandardScaler(), num_cols),
                          ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)])
Xtr = prep.fit_transform(X_train); Xte = prep.transform(X_test)
print("train:", Xtr.shape, " test:", Xte.shape)

### A2 · Four classical models
| Model | Idea |
|---|---|
| Logistic regression | linear decision boundary, the epidemiologist's workhorse |
| Random forest | many decorrelated decision trees vote |
| Gradient boosting | trees are added one by one, each correcting the errors of the previous ones (XGBoost/LightGBM family) |
| Neural network (MLP) | layers of weighted sums + non-linearities, trained with backpropagation |

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score

models = {
    "Logistic regression": LogisticRegression(max_iter=2000),
    "Random forest":       RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=0, n_jobs=-1),
    "Gradient boosting":   HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=300, random_state=0),
    "Neural net (MLP)":    MLPClassifier(hidden_layer_sizes=(32, 16), alpha=1e-3, early_stopping=True, max_iter=500, random_state=0),
}
results = {}
for name, m in models.items():
    # ✏️ TODO: fit the model on the TRAINING data  (hint: m.fit(features, labels))
    m.fit(___)
    p_test = m.predict_proba(Xte)[:, 1]
    results[name] = roc_auc_score(y_test, p_test)
pd.Series(results, name="test AUC").round(3).sort_values(ascending=False)

### A3 · Overfitting in one picture
A random forest whose trees may grow **without limit** memorises the training data.
Compare the AUC on the training data with the AUC on the test data.

In [ ]:
rows = []
# ✏️ TODO: add a large leaf size (e.g. 50 or 100) to see how it controls overfitting
for leaf in [1, 5, 20, ___]:
    rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=leaf, random_state=0, n_jobs=-1).fit(Xtr, y_train)
    rows.append(dict(min_samples_leaf=leaf,
                     train_AUC=roc_auc_score(y_train, rf.predict_proba(Xtr)[:, 1]),
                     test_AUC=roc_auc_score(y_test, rf.predict_proba(Xte)[:, 1])))
pd.DataFrame(rows).round(3)

**❓ Question:** With `min_samples_leaf=1` the training AUC is ≈ 1.0. Why is that *not* good news?

### A4 · TabPFN: a foundation model for tables (optional, ≈ 2 min)
TabPFN (Hollmann et al., *Nature* 2025) was pre-trained on millions of **synthetic** datasets. It does not "train" on your data in the usual sense: it reads the training set as context and predicts in one forward pass (*in-context learning*, like an LLM).

We use the openly available **v2 weights**, which download without a PriorLabs account. Newer versions (2.5, 3.x) require accepting a licence in the browser.

In [ ]:
%pip -q install tabpfn
try:
    from tabpfn import TabPFNClassifier
    from tabpfn.constants import ModelVersion
    tab = TabPFNClassifier.create_default_for_version(ModelVersion.V2, device=DEVICE)
    n_ctx = 3000 if DEVICE == "cuda" else 1000          # CPU is limited to ~1000 training samples
    Xd_tr = Xtr.toarray() if hasattr(Xtr, "toarray") else Xtr
    Xd_te = Xte.toarray() if hasattr(Xte, "toarray") else Xte
    tab.fit(Xd_tr[:n_ctx], y_train[:n_ctx])
    results["TabPFN v2 (foundation model)"] = roc_auc_score(y_test, tab.predict_proba(Xd_te)[:, 1])
except Exception as e:
    print("TabPFN could not run here:", type(e).__name__, str(e)[:300])
pd.Series(results, name="test AUC").round(3).sort_values(ascending=False)

**💡 Take-away Part A:** On tabular data with a few thousand rows, the differences between models are small, and a well-specified logistic regression is often just as good. Boosting and TabPFN are the strongest *general-purpose* choices. Deep networks shine on **images, text and signals**, which is where we go now.

---
# Part B: Your first CNN on abdominal MRI

**Data:** [CHAOS challenge](https://chaos.grand-challenge.org/), 20 patients, T1-weighted **in-phase / opposed-phase** MRI (a 2-point Dixon sequence) with expert masks of liver, kidneys and spleen. Licence CC BY-NC-SA 4.0 (Kavur et al., *Med Image Anal* 2021).

**Task (classification):** *Is at least one kidney visible in this axial slice?*

In [ ]:
#@title 🔧 Helper functions: download & load the CHAOS MRI data (just run this cell) { display-mode: "form" }
# CHAOS challenge data (Kavur et al., Med Image Anal 2021), Zenodo record 3431873, licence CC BY-NC-SA 4.0.
# We use the 20 MR training cases, T1-DUAL sequence: in-phase (IP) and opposed-phase (OP) images
# plus expert masks of liver, right kidney, left kidney and spleen.
import os, io, glob, time, zipfile, urllib.request, shutil
from concurrent.futures import ThreadPoolExecutor
import numpy as np

CHAOS_URL = "https://zenodo.org/records/3431873/files/CHAOS_Train_Sets.zip?download=1"
# Colab: /content/chaos · locally: ~/yn_data/chaos (shared by all notebooks, outside Dropbox/git); override with YN_CHAOS_ROOT
CHAOS_DIR = os.environ.get("YN_CHAOS_ROOT", "/content/chaos" if os.path.isdir("/content") else os.path.expanduser("~/yn_data/chaos"))
LABELS = {1: "liver", 2: "kidney_right", 3: "kidney_left", 4: "spleen"}


def _get(url, start, end, tries=6):
    """Bytes start..end of a web file, retrying dropped connections (Zenodo is sometimes slow/flaky)."""
    for k in range(tries):
        try:
            req = urllib.request.Request(url, headers={"Range": f"bytes={start}-{end}"})
            with urllib.request.urlopen(req, timeout=60) as r:
                if r.status != 206:
                    raise IOError("server ignores byte ranges")
                data = r.read()
            if len(data) == end - start + 1:
                return data
        except Exception as e:
            if "ignores byte ranges" in str(e) or k == tries - 1:
                raise
        time.sleep(2 * (k + 1))
    raise IOError(f"incomplete download of bytes {start}-{end}")


class _RemoteZip(io.RawIOBase):
    """Seekable read-only view of a zip on a web server: only the byte ranges that are read get downloaded."""
    def __init__(self, url):
        self.url, self.pos, self.chunks = url, 0, {}
        with urllib.request.urlopen(urllib.request.Request(url, method="HEAD"), timeout=60) as r:
            self.size = int(r.headers["Content-Length"])
    def readable(self): return True
    def seekable(self): return True
    def tell(self): return self.pos
    def seek(self, off, whence=0):
        self.pos = (off, self.pos + off, self.size + off)[whence]
        return self.pos
    def prefetch(self, start, end):
        self.chunks[start] = _get(self.url, start, min(end, self.size - 1))
    def read(self, n=-1):
        n = self.size - self.pos if n is None or n < 0 else min(n, self.size - self.pos)
        if n <= 0:
            return b""
        for s, d in self.chunks.items():
            if s <= self.pos and self.pos + n <= s + len(d):
                out = d[self.pos - s:self.pos - s + n]
                break
        else:
            out = _get(self.url, self.pos, self.pos + n - 1)
        self.pos += n
        return out


def _is_mr_t1dual(name):
    name = name.replace("\\", "/")
    return "/MR/" in name and "/T1DUAL/" in name and not name.endswith("/")


def _download_mr_only(dest):
    """Fetch only the MR T1-DUAL files (≈ 65 MB) out of the 0.9 GB zip, in ~1 MB blocks, 8 in parallel."""
    rz = _RemoteZip(CHAOS_URL)
    z = zipfile.ZipFile(rz)
    members = sorted((i for i in z.infolist() if _is_mr_t1dual(i.filename)), key=lambda i: i.header_offset)
    spans = []                                    # merge neighbouring files into larger download blocks
    for i in members:
        s, e = i.header_offset, i.header_offset + 30 + len(i.filename.encode()) + len(i.extra) + i.compress_size + 1024
        if spans and s - spans[-1][1] < (1 << 18) and e - spans[-1][0] < (4 << 20):
            spans[-1][1] = e
        else:
            spans.append([s, e])
    print(f"Downloading {len(members)} MR files (≈ {sum(i.compress_size for i in members) / 1e6:.0f} MB) from Zenodo ...")
    done = [0]
    def fetch(span):
        rz.prefetch(*span)
        done[0] += 1
        if done[0] % 10 == 0 or done[0] == len(spans):
            print(f"  {done[0]}/{len(spans)} blocks")
    with ThreadPoolExecutor(8) as ex:
        list(ex.map(fetch, spans))
    z.extractall(dest, members)


def _download_full(dest):
    """Fallback: the whole zip (≈ 0.9 GB), resuming after dropped connections."""
    zpath = dest.rstrip("/") + "_download.zip"
    with urllib.request.urlopen(urllib.request.Request(CHAOS_URL, method="HEAD"), timeout=60) as r:
        size = int(r.headers["Content-Length"])
    print("Downloading the full CHAOS training zip (≈ 0.9 GB) ...")
    for k in range(20):
        have = os.path.getsize(zpath) if os.path.exists(zpath) else 0
        if have >= size:
            break
        try:
            req = urllib.request.Request(CHAOS_URL, headers={"Range": f"bytes={have}-"} if have else {})
            with urllib.request.urlopen(req, timeout=60) as r, open(zpath, "ab" if r.status == 206 else "wb") as f:
                shutil.copyfileobj(r, f, length=1 << 22)
        except Exception as e:
            print(f"  connection dropped ({type(e).__name__}), resuming at {have / 1e6:.0f} MB ...")
            time.sleep(5)
    with zipfile.ZipFile(zpath) as z:
        z.extractall(dest, [m for m in z.namelist() if _is_mr_t1dual(m)])
    os.remove(zpath)


def download_chaos(dest=CHAOS_DIR, force=False):
    """Download and unpack the 20 MR cases (T1-DUAL) of the CHAOS training set.
    Safe to re-run: complete data is reused, leftovers of an interrupted download are deleted first."""
    marker = os.path.join(dest, ".download_complete")
    if os.path.exists(marker) and not force:
        print("CHAOS data already available in", dest)
        return dest
    if os.path.isdir(dest):                       # incomplete/old download -> start clean (else slice counts/shapes mismatch)
        shutil.rmtree(dest)
    os.makedirs(dest)
    try:
        _download_mr_only(dest)
    except Exception as e:
        print(f"Partial download not possible ({type(e).__name__}: {str(e)[:100]})")
        shutil.rmtree(dest); os.makedirs(dest)
        _download_full(dest)
    open(marker, "w").close()
    print("Done:", len(glob.glob(os.path.join(dest, "**", "T1DUAL"), recursive=True)), "MR cases in", dest)
    return dest


def _png_to_labels(png):
    """CHAOS ground-truth PNGs encode organs as grey values: liver 63, R kidney 126, L kidney 189, spleen 252."""
    lab = np.zeros(png.shape, np.uint8)
    lab[(png >= 55) & (png <= 70)] = 1
    lab[(png >= 110) & (png <= 135)] = 2
    lab[(png >= 175) & (png <= 200)] = 3
    lab[(png >= 240)] = 4
    return lab


def load_case(case_dir):
    """Load one CHAOS T1-DUAL case.
    Returns dict with 'ip', 'op' (float32, z×y×x), 'mask' (uint8, z×y×x),
    'spacing' (x, y, z in mm), 'origin', 'direction' (for building a 3D image), 'id'."""
    import pydicom
    from PIL import Image
    t1 = os.path.join(case_dir, "T1DUAL")
    ip_files = sorted(glob.glob(os.path.join(t1, "DICOM_anon", "InPhase", "*.dcm")))
    op_files = sorted(glob.glob(os.path.join(t1, "DICOM_anon", "OutPhase", "*.dcm")))
    gt_files = sorted(glob.glob(os.path.join(t1, "Ground", "*.png")))
    n = min(len(ip_files), len(op_files), len(gt_files))
    if not (len(ip_files) == len(op_files) == len(gt_files)):
        print(f"  note: case {os.path.basename(case_dir)} has {len(ip_files)}/{len(op_files)}/{len(gt_files)} IP/OP/GT slices, using {n}")
    ip_ds = [pydicom.dcmread(f) for f in ip_files[:n]]
    op_ds = [pydicom.dcmread(f) for f in op_files[:n]]
    gt = [_png_to_labels(np.array(Image.open(f).convert("L"))) for f in gt_files[:n]]
    # geometry from DICOM header -> sort slices along the slice normal
    iop = np.array(ip_ds[0].ImageOrientationPatient, float)
    r, c = iop[:3], iop[3:]
    nrm = np.cross(r, c)
    pos = np.array([np.dot(np.array(d.ImagePositionPatient, float), nrm) for d in ip_ds])
    order = np.argsort(pos)
    ip = np.stack([ip_ds[i].pixel_array.astype(np.float32) for i in order])
    op = np.stack([op_ds[i].pixel_array.astype(np.float32) for i in order])
    mask = np.stack([gt[i] for i in order])
    dz = float(np.median(np.diff(pos[order]))) if n > 1 else float(ip_ds[0].SliceThickness)
    ps = [float(v) for v in ip_ds[0].PixelSpacing]  # [row spacing, column spacing]
    direction = (r[0], c[0], nrm[0], r[1], c[1], nrm[1], r[2], c[2], nrm[2])
    origin = tuple(float(v) for v in ip_ds[order[0]].ImagePositionPatient)
    return dict(ip=ip, op=op, mask=mask, spacing=(ps[1], ps[0], abs(dz)), origin=origin,
                direction=direction, id=os.path.basename(case_dir))


def load_all_cases(root=CHAOS_DIR):
    case_dirs = sorted({os.path.dirname(p) for p in glob.glob(os.path.join(root, "**", "MR", "*", "T1DUAL"), recursive=True)},
                       key=lambda s: int(os.path.basename(s)) if os.path.basename(s).isdigit() else s)
    cases = [load_case(d) for d in case_dirs]
    print(f"Loaded {len(cases)} cases, {sum(c['ip'].shape[0] for c in cases)} slices per contrast")
    return cases


def to_sitk(vol, case):
    """Wrap a z×y×x numpy array into a SimpleITK image with the case geometry."""
    import SimpleITK as sitk
    img = sitk.GetImageFromArray(vol)
    img.SetSpacing(case["spacing"]); img.SetOrigin(case["origin"]); img.SetDirection(case["direction"])
    return img


def _norm(vol):
    lo, hi = np.percentile(vol, 1), np.percentile(vol, 99.5)
    return np.clip((vol - lo) / (hi - lo + 1e-6), 0, 1).astype(np.float32)


def make_slice_dataset(cases, size=128, min_kidney_px=50):
    """Turn the 3D cases into a 2D slice dataset.
    X_ip, X_op : (N, size, size) images scaled to [0, 1] (per volume)
    y_kidney   : 1 if at least one kidney is visible in the slice
    z_rel      : relative slice position in the volume (0 = lowest slice, 1 = highest)
    patient    : case id (for patient-level splitting!)
    organs     : list of visible organs per slice (used for captions)"""
    import torch, torch.nn.functional as F
    X_ip, X_op, M, yk, zr, pid, organs = [], [], [], [], [], [], []
    for c in cases:
        ipn, opn = _norm(c["ip"]), _norm(c["op"])
        nz = ipn.shape[0]
        for z in range(nz):
            m = c["mask"][z]
            X_ip.append(ipn[z]); X_op.append(opn[z]); M.append(m)
            k = int(((m == 2) | (m == 3)).sum() >= min_kidney_px)
            yk.append(k); zr.append(z / max(nz - 1, 1)); pid.append(c["id"])
            organs.append([LABELS[l] for l in (1, 2, 3, 4) if (m == l).sum() >= min_kidney_px])
    def rs(a, mode):
        t = torch.from_numpy(np.stack(a)[:, None].astype(np.float32))
        t = F.interpolate(t, size=(size, size), mode=mode, **({} if mode == "nearest" else {"align_corners": False}))
        return t[:, 0].numpy()
    X_ip, X_op = rs(X_ip, "bilinear"), rs(X_op, "bilinear")
    M = rs(M, "nearest").astype(np.uint8)
    return dict(X_ip=X_ip, X_op=X_op, mask=M, y_kidney=np.array(yk), z_rel=np.array(zr, np.float32),
                patient=np.array(pid), organs=organs)


def patient_split(patients, n_val=3, n_test=4, seed=0):
    """Split by PATIENT (never by slice!) into train/val/test index arrays."""
    ids = np.array(sorted(set(patients), key=lambda s: int(s) if str(s).isdigit() else s))
    rng = np.random.default_rng(seed); rng.shuffle(ids)
    test_ids, val_ids, train_ids = ids[:n_test], ids[n_test:n_test + n_val], ids[n_test + n_val:]
    idx = lambda s: np.where(np.isin(patients, s))[0]
    return idx(train_ids), idx(val_ids), idx(test_ids)

In [ ]:
download_chaos()
cases = load_all_cases()
data = make_slice_dataset(cases, size=128)
X, yk = data["X_ip"], data["y_kidney"]
print("slices:", X.shape, "| with kidney:", yk.sum())

fig, ax = plt.subplots(2, 6, figsize=(14, 5))
for i, a in enumerate(ax.flat):
    j = np.random.RandomState(i).randint(len(X))
    a.imshow(X[j], cmap="gray"); a.set_title(f"pat {data['patient'][j]} · kidney={yk[j]}", fontsize=9); a.axis("off")
plt.tight_layout()

### B1 · Split by **patient**, not by slice
Neighbouring slices of the same patient look almost identical. If they end up in both training and test set, the test result is too optimistic (*data leakage*; we'll measure this effect in Hands-on 3).

In [ ]:
tr, va, te = patient_split(data["patient"], n_val=3, n_test=4, seed=0)
ids = lambda idx: sorted({str(p) for p in data["patient"][idx]}, key=int)
print("train patients:", ids(tr), "\nval patients:  ", ids(va), "\ntest patients: ", ids(te))
print("slices train/val/test:", len(tr), len(va), len(te))

### B2 · The network: a chain of learned filters
Four blocks of *convolution → batch normalisation → ReLU → max-pooling*, then global average pooling and one output neuron (probability of "kidney visible").

In [ ]:
import torch.nn as nn, torch.nn.functional as F

class SmallCNN(nn.Module):
    def __init__(self, channels=(16, 32, 64, 64), dropout=0.0):
        super().__init__()
        layers, c_in = [], 1
        for c in channels:
            layers += [nn.Conv2d(c_in, c, 3, padding=1), nn.BatchNorm2d(c), nn.ReLU(), nn.MaxPool2d(2)]
            c_in = c
        self.features = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(dropout), nn.Linear(c_in, 1))
    def forward(self, x):
        return self.head(self.features(x)).squeeze(1)      # logits

model = SmallCNN()
print(model)
print("trainable parameters:", sum(p.numel() for p in model.parameters()))

### B3 · Training loop: forward → loss → backward → update
This is exactly the weight update from the lecture slide, repeated for every mini-batch.

In [ ]:
def augment(xb):
    # random left-right flip + random brightness/contrast (data augmentation)
    flip = torch.rand(xb.shape[0], device=xb.device) < 0.5
    xb = torch.where(flip[:, None, None, None], xb.flip(-1), xb)
    gain = 1 + 0.2 * (torch.rand(xb.shape[0], 1, 1, 1, device=xb.device) - 0.5)
    return xb * gain

def train_cnn(model, Xtr_, ytr_, Xva_, yva_, epochs=15, lr=1e-3, batch=32, use_aug=True):
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    Xt, yt = torch.tensor(Xtr_[:, None]), torch.tensor(ytr_, dtype=torch.float32)
    Xv, yv = torch.tensor(Xva_[:, None]).to(DEVICE), torch.tensor(yva_, dtype=torch.float32).to(DEVICE)
    hist = {"train_loss": [], "val_loss": []}
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(Xt)); run = 0.0
        for i in range(0, len(Xt), batch):
            idx = perm[i:i + batch]
            xb, yb = Xt[idx].to(DEVICE), yt[idx].to(DEVICE)
            if use_aug: xb = augment(xb)
            logits = model(xb)                                            # 1. forward pass
            loss = F.binary_cross_entropy_with_logits(logits, yb)         # 2. loss (cross-entropy)
            opt.zero_grad()
            # ✏️ TODO: 3. backpropagate the error  (hint: call .backward() on the loss)
            ___
            # ✏️ TODO: 4. update the weights  (hint: the optimizer takes a .step())
            ___
            run += loss.item() * len(idx)
        model.eval()
        with torch.no_grad():
            vl = F.binary_cross_entropy_with_logits(model(Xv), yv).item()
        hist["train_loss"].append(run / len(Xt)); hist["val_loss"].append(vl)
        print(f"epoch {ep+1:2d}  train loss {hist['train_loss'][-1]:.3f}  val loss {vl:.3f}")
    return hist

@torch.no_grad()
def predict(model, X_):
    model.eval().to(DEVICE)
    return torch.cat([torch.sigmoid(model(torch.tensor(X_[i:i+256, None]).to(DEVICE))).cpu() for i in range(0, len(X_), 256)]).numpy()

def plot_hist(h, title=""):
    plt.figure(figsize=(5, 3)); plt.plot(h["train_loss"], label="train"); plt.plot(h["val_loss"], label="validation")
    plt.xlabel("epoch"); plt.ylabel("loss"); plt.title(title); plt.legend(); plt.show()

In [ ]:
torch.manual_seed(0)
model = SmallCNN()
hist = train_cnn(model, X[tr], yk[tr], X[va], yk[va], epochs=15, lr=1e-3)
plot_hist(hist, "SmallCNN, lr=1e-3, with augmentation")
p_te = predict(model, X[te])
print(f"TEST  AUC = {roc_auc_score(yk[te], p_te):.3f}   accuracy = {((p_te > 0.5) == yk[te]).mean():.3f}")

In [ ]:
# Look at the predictions for a few test slices
fig, ax = plt.subplots(2, 6, figsize=(14, 5))
pick = np.random.RandomState(1).choice(len(te), 12, replace=False)
for a, j in zip(ax.flat, pick):
    k = te[j]; ok = (p_te[j] > 0.5) == yk[k]
    a.imshow(X[k], cmap="gray"); a.axis("off")
    a.set_title(f"p(kidney)={p_te[j]:.2f} | true={yk[k]}", fontsize=9, color="green" if ok else "red")
plt.tight_layout()

### B4 · Experiments (pick at least two)
Change **one** thing at a time and re-run the cell. Note the test AUC.

| # | Experiment | What to change |
|---|---|---|
| 1 | Learning rate too high / too low | `lr=1e-1` or `lr=1e-5` |
| 2 | Overfitting | `epochs=40, use_aug=False` → watch the validation loss |
| 3 | Regularisation | `SmallCNN(dropout=0.5)` together with experiment 2 |
| 4 | Smaller network | `SmallCNN(channels=(4, 8))` |

In [ ]:
torch.manual_seed(0)
# ✏️ TODO: choose your experiment (see table above)
model_exp = SmallCNN(dropout=___)
hist_exp = train_cnn(model_exp, X[tr], yk[tr], X[va], yk[va], epochs=___, lr=___, use_aug=___)
plot_hist(hist_exp, "your experiment")
print(f"TEST AUC = {roc_auc_score(yk[te], predict(model_exp, X[te])):.3f}")

### B5 · What did the network learn? First-layer filters
The first convolution layer learns small 3×3 filters, many of which look like the edge and blob detectors from classical image processing.

In [ ]:
w = model.features[0].weight.detach().cpu()[:, 0]
fig, ax = plt.subplots(2, 8, figsize=(10, 3))
for a, k in zip(ax.flat, w):
    a.imshow(k, cmap="RdBu_r"); a.axis("off")
plt.suptitle("16 learned 3×3 filters of layer 1"); plt.show()

---
### ✅ Summary
* Classical ML (logistic regression, random forests, boosting) and **TabPFN** are strong for tabular data.
* A CNN learns its own filters from pixels: **forward → loss → backward → update**.
* Always split by **patient**, monitor the **validation loss**, and use augmentation/regularisation against overfitting.

➡️ After the break: **Beyond tables**, with segmentation, foundation models, embeddings and CLIP.